# ScoreCard 完整演示样例

展示 ScoreCard 的各种使用方式：
1. 从零开始训练
2. 使用预训练 LR 模型
3. 使用 lr_kwargs 参数
4. 评分卡刻度表
5. 模型评估

In [ ]:
import sys
sys.path.insert(0, '/Users/xiaoxi/CodeBuddy/hscredit/hscredit')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from hscredit.core.models import ScoreCard, LogisticRegression
from hscredit.core.binning import OptimalBinning
from hscredit.core.metrics import KS, AUC
from hscredit.utils.datasets import germancredit

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("ScoreCard 演示样例初始化完成！")

## 1. 加载数据

In [ ]:
# 加载 German Credit 数据集
df = germancredit()
df['class'] = df['class'].astype(int)

# 分离特征和目标
target = 'class'
feature_cols = [c for c in df.columns if c != target]

print(f"数据形状: {df.shape}")
print(f"特征数: {len(feature_cols)}")
print(f"\n目标分布:")
print(df[target].value_counts())

# 划分训练集和测试集
train, test = train_test_split(df, test_size=0.3, random_state=42, stratify=df[target])
print(f"\n训练集: {train.shape[0]} 样本")
print(f"测试集: {test.shape[0]} 样本")

## 2. 数据预处理 - 分箱和 WOE 转换

In [ ]:
# 最优分箱
binner = OptimalBinning(max_n_bins=5, method='chi_merge')
binner.fit(train[feature_cols], train[target])

# WOE 转换
train_woe = binner.transform(train[feature_cols])
test_woe = binner.transform(test[feature_cols])

print(f"WOE 转换后特征数: {train_woe.shape[1]}")
print(f"\nWOE 数据预览:")
train_woe.head()

## 3. 方式1 - 从零开始训练 ScoreCard

In [ ]:
# 创建并训练 ScoreCard
scorecard1 = ScoreCard(
    pdo=50,
    rate=2,
    base_odds=1/19,  # 对应约 5% 的坏样本率
    base_score=750,
    calculate_stats=True
)

scorecard1.fit(train_woe, train[target])

print(f"ScoreCard 训练完成！")
print(f"入模特征数: {len(scorecard1.feature_names_)}")
print(f"模型截距: {scorecard1.intercept_:.4f}")

In [ ]:
# 预测评分
train_scores1 = scorecard1.predict(train_woe)
test_scores1 = scorecard1.predict(test_woe)

print(f"训练集分数范围: [{train_scores1.min():.2f}, {train_scores1.max():.2f}]")
print(f"训练集分数均值: {train_scores1.mean():.2f}")
print(f"测试集分数范围: [{test_scores1.min():.2f}, {test_scores1.max():.2f}]")
print(f"测试集分数均值: {test_scores1.mean():.2f}")

In [ ]:
# 查看评分卡规则（前5个特征）
for i, (feature, rule) in enumerate(list(scorecard1.rules_.items())[:5]):
    print(f"\n特征 {i+1}: {feature}")
    print(f"  系数: {rule['coef']:.4f}")
    if rule['bins'] is not None:
        print(f"  分箱数: {len(rule['bins'])}")
    if rule['woe'] is not None:
        print(f"  WOE值: {rule['woe']}")
    if rule['scores'] is not None:
        print(f"  分数: {rule['scores']}")

## 4. 方式2 - 使用预训练 LR 模型

In [ ]:
# 训练逻辑回归模型
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(train_woe, train[target])

print(f"逻辑回归模型训练完成！")
print(f"模型系数（前5个）: {lr_model.coef_[0][:5]}")
print(f"模型截距: {lr_model.intercept_[0]:.4f}")

In [ ]:
# 使用预训练模型创建 ScoreCard（不调用 fit）
scorecard2 = ScoreCard(
    binner=binner,
    lr_model=lr_model,
    pdo=50,
    rate=2,
    base_odds=1/19,
    base_score=750
)

print(f"ScoreCard 创建完成！")
print(f"lr_model_ 是否已设置: {scorecard2.lr_model_ is not None}")

In [ ]:
# 直接预测（不调用 fit）
train_scores2 = scorecard2.predict(train_woe)
test_scores2 = scorecard2.predict(test_woe)

print(f"✅ 预测成功！")
print(f"训练集分数范围: [{train_scores2.min():.2f}, {train_scores2.max():.2f}]")
print(f"训练集分数均值: {train_scores2.mean():.2f}")
print(f"测试集分数范围: [{test_scores2.min():.2f}, {test_scores2.max():.2f}]")
print(f"测试集分数均值: {test_scores2.mean():.2f}")

In [ ]:
# 验证两种方式结果一致
diff = np.abs(train_scores1 - train_scores2).max()
print(f"两种方式预测结果最大差异: {diff:.6f}")
if diff < 1e-6:
    print(f"✅ 结果完全一致！")
else:
    print(f"⚠️ 结果有差异，但可能在可接受范围内")

## 5. 方式3 - 使用 lr_kwargs 参数

In [ ]:
# 使用 lr_kwargs 传入逻辑回归参数
scorecard3 = ScoreCard(
    pdo=50,
    rate=2,
    base_odds=1/19,
    base_score=750,
    lr_kwargs={'C': 0.1, 'max_iter': 500, 'random_state': 42}
)

scorecard3.fit(train_woe, train[target])
print(f"ScoreCard 训练完成！")

train_scores3 = scorecard3.predict(train_woe)
print(f"训练集分数均值: {train_scores3.mean():.2f}")

## 6. 评分卡刻度表

In [ ]:
# 查看评分卡刻度表
scale_df = scorecard1.scorecard_scale()
print("评分卡刻度参数：")
print(scale_df.to_string(index=False))

## 7. 模型评估

In [ ]:
# 预测概率
train_proba = scorecard1.predict_proba(train_woe)[:, 1]
test_proba = scorecard1.predict_proba(test_woe)[:, 1]

# 计算 KS
train_ks = KS(train[target].values, train_proba)
test_ks = KS(test[target].values, test_proba)

# 计算 AUC
train_auc = AUC(train[target].values, train_proba)
test_auc = AUC(test[target].values, test_proba)

print(f"训练集 KS: {train_ks:.4f}")
print(f"测试集 KS: {test_ks:.4f}")
print(f"\n训练集 AUC: {train_auc:.4f}")
print(f"测试集 AUC: {test_auc:.4f}")

In [ ]:
# 绘制分数分布图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 训练集分数分布
axes[0].hist(train_scores1[train[target] == 0], bins=30, alpha=0.6, label='好客户', color='green')
axes[0].hist(train_scores1[train[target] == 1], bins=30, alpha=0.6, label='坏客户', color='red')
axes[0].set_xlabel('评分')
axes[0].set_ylabel('频数')
axes[0].set_title('训练集分数分布')
axes[0].legend()

# 测试集分数分布
axes[1].hist(test_scores1[test[target] == 0], bins=30, alpha=0.6, label='好客户', color='green')
axes[1].hist(test_scores1[test[target] == 1], bins=30, alpha=0.6, label='坏客户', color='red')
axes[1].set_xlabel('评分')
axes[1].set_ylabel('频数')
axes[1].set_title('测试集分数分布')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. 保存和导出

In [ ]:
# 查看评分卡规则
print(f"评分卡规则数: {len(scorecard1.rules_)}")
print(f"\n入模特征（前10个）:")
for i, feature in enumerate(scorecard1.feature_names_[:10]):
    print(f"  {i+1}. {feature}")

In [ ]:
# 导出 PMML（可选）
# scorecard1.export_pmml('scorecard.pmml')
# print("PMML 文件已导出！")

## 9. 总结

本演示展示了 ScoreCard 的多种使用方式：

1. **从零开始训练**: 创建 ScoreCard 后调用 fit 方法
2. **使用预训练模型**: 传入 lr_model，无需调用 fit 即可预测
3. **使用 lr_kwargs**: 通过参数控制逻辑回归模型

ScoreCard 支持的功能：
- 评分卡刻度转换
- 评分预测
- 概率预测
- 评分卡规则导出
- PMML 导出